<a href="https://colab.research.google.com/github/chinl00n/capstone/blob/main/Sustain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

# GitHub raw file URL
base_url = "https://raw.githubusercontent.com/chinl00n/capstone/main/"

# Load datasets directly from GitHub
sustainability = pd.read_csv(
    base_url + "Sustainability%20Dataset.csv"
)

country = pd.read_excel(base_url + "Country.xlsx")
continent = pd.read_excel(base_url + "Continent.xlsx")
energy = pd.read_excel(base_url + "Energy.xlsx")
electricity = pd.read_excel(base_url + "Electricity.xlsx")

# Validate unique country-year records
key = ['Country Code', 'Year']
assert not sustainability.duplicated(key).any()
assert not energy.duplicated(key).any()
assert not electricity.duplicated(key).any()

# Clean string values to remove any trailing/leading whitespaces
for df_temp in [country, continent]:
    for col in df_temp.select_dtypes(include='object').columns:
        df_temp[col] = df_temp[col].astype(str).str.strip()

# Check lookup consistency before deduplicating (allowing minor inconsistencies by taking the first unique non-null mapping)
country_lookup = country.drop_duplicates('Country Code')
continent_lookup = continent.drop_duplicates('Country Code')

# Attach geographic metadata without multiplying country-year rows
df = sustainability.merge(country_lookup, on='Country Code', how='left', validate='many_to_one')
df = df.merge(continent_lookup, on='Country Code', how='left', validate='many_to_one')
assert len(df) == len(sustainability)

# Supplementary energy/electricity data are already represented in the main
# dataset; inspect and reconcile them before deciding whether to merge.

# Report missing values
missing = df.isna().mean().mul(100).sort_values(ascending=False)
print('Missing data (%):')
print(missing.head(10).round(1))

# Compare median GDP per capita by continent
metric = 'GDP per capita (current US$) - NY.GDP.PCAP.CD'
comparison = df.groupby('Continent')[metric].median().sort_values(ascending=False)
print('\nMedian country-year GDP per capita by continent:')
print(comparison.round(2))

Missing data (%):
Gini index (World Bank estimate) - SI.POV.GINI                                                  60.4
Proportion of population below international poverty line (%) - SI_POV_DAY1 - 1.1.1             60.3
Proportion of population covered by at least a 3G mobile network (%) - IT_MOB_3GNTWK - 9.c.1    58.7
Unemployment rate, women (%) - SL_TLF_UEM - 8.5.2                                               45.1
Unemployment rate, male (%) - SL_TLF_UEM - 8.5.2                                                45.0
Prevalence of undernourishment (%) - SN_ITK_DEFC - 2.1.1                                        44.2
Proportion of population using basic drinking water services (%) - SP_ACS_BSRVH2O - 1.4.1       44.1
Children out of school (% of primary school age) - SE.PRM.UNER.ZS                               32.3
Primary completion rate, total (% of relevant age group) - SE.PRM.CMPT.ZS                       32.3
Automated teller machines (ATMs) (per 100,000 adults) - FB.ATM.TOTL.P5   